# python_dotenv: 5 Basic Usage Examples

In [ ]:
# Example 1: Load environment variables from a file
import os
import tempfile
from dotenv import load_dotenv

env_content = """# Sample environment variables
DATABASE_URL=sqlite:///app.db
DEBUG=True
SECRET_KEY=your-secret-key-here
"""

# Start from a clean slate for these three names, so the result does not depend on the caller's environment
for key in ("DATABASE_URL", "DEBUG", "SECRET_KEY"):
    os.environ.pop(key, None)

try:
    # Write the sample file into a temporary directory, never into the working directory
    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, ".env")
        with open(path, "w") as f:
            f.write(env_content)

        # Load environment variables
        assert load_dotenv(path)

    print("Environment variables loaded:")
    print(f'DATABASE_URL: {os.getenv("DATABASE_URL", "Not found")}')
    print(f'DEBUG: {os.getenv("DEBUG", "Not found")}')
    print(f'SECRET_KEY: {os.getenv("SECRET_KEY", "Not found")}')
    assert os.environ["DATABASE_URL"] == "sqlite:///app.db" and os.environ["DEBUG"] == "True"
    assert os.environ["SECRET_KEY"] == "your-secret-key-here"
finally:
    # Remove the three variables again, so later cells start from the same environment
    for key in ("DATABASE_URL", "DEBUG", "SECRET_KEY"):
        os.environ.pop(key, None)

assert "DATABASE_URL" not in os.environ

In [ ]:
# Example 2: Read variables from a string, without touching os.environ or any file
import io
import os
from dotenv import dotenv_values

text = "APP_HOST=localhost\nAPP_PORT=8080\nAPP_URL=http://${APP_HOST}:${APP_PORT}\nAPP_EMPTY=\n"
env = dotenv_values(stream=io.StringIO(text))
print(env)
assert env == {"APP_HOST": "localhost", "APP_PORT": "8080", "APP_URL": "http://localhost:8080", "APP_EMPTY": ""}, env
assert "APP_URL" not in os.environ   # dotenv_values only returns a dict

In [ ]:
# Example 3: Overriding existing environment variables
import os
import tempfile
from dotenv import load_dotenv

with tempfile.TemporaryDirectory() as td:
    path = os.path.join(td, ".env")
    with open(path, "w") as f:
        f.write("FOO=bar\n")

    os.environ["FOO"] = "baz"
    load_dotenv(path)                  # by default a variable that is already set is kept
    print("without override:", os.getenv("FOO"))
    assert os.environ["FOO"] == "baz"

    load_dotenv(path, override=True)   # override=True lets the file win
    print("with override=True:", os.getenv("FOO"))
    assert os.environ["FOO"] == "bar"

del os.environ["FOO"]

In [ ]:
# Example 4: Edit a .env file with set_key, get_key and unset_key
import os
import tempfile
from dotenv import dotenv_values, get_key, set_key, unset_key

with tempfile.TemporaryDirectory() as td:
    path = os.path.join(td, "demo.env")
    with open(path, "w") as f:
        f.write("# Demo environment file\nAPI_KEY=demo123\n")
    print("Created", os.path.basename(path), "->", dotenv_values(path))

    set_key(path, "LOG_LEVEL", "debug")
    assert get_key(path, "LOG_LEVEL") == "debug"
    print("After set_key:", dotenv_values(path))

    unset_key(path, "API_KEY")
    assert dotenv_values(path) == {"LOG_LEVEL": "debug"}, dotenv_values(path)
    print("After unset_key:", dotenv_values(path))

assert not os.path.exists(path)
print("The temporary directory and its demo.env file are gone")

In [ ]:
# Example 5: Read typed configuration values from a .env file
import importlib.metadata
import os
import tempfile

import dotenv
from dotenv import load_dotenv

print(f"python-dotenv {importlib.metadata.version('python-dotenv')} (module {dotenv.__name__})")

config_keys = ("DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DEBUG", "API_KEY")
# Start from a clean slate for these names, so the result does not depend on the caller's environment
for key in config_keys:
    os.environ.pop(key, None)

with tempfile.TemporaryDirectory() as td:
    path = os.path.join(td, ".env")
    # Create a sample .env file for demonstration
    with open(path, "w") as f:
        f.write("""# Database configuration
DB_HOST=localhost
DB_PORT=5432
DB_NAME=myapp
DB_USER=admin
DEBUG=True
API_KEY=demo-api-key
""")

    # Load environment variables from the .env file
    load_dotenv(path)

# Every value arrives as a string; convert to the type the application needs
db_host = os.getenv("DB_HOST", "default_host")
db_port = int(os.getenv("DB_PORT", 3306))
debug_mode = os.getenv("DEBUG", "False").lower() == "true"
api_key = os.getenv("API_KEY")

print(f"Database: {db_host}:{db_port}")
print(f"Debug mode: {debug_mode}")
print(f"API key loaded: {'Yes' if api_key else 'No'}")
assert (db_host, db_port, debug_mode) == ("localhost", 5432, True), (db_host, db_port, debug_mode)
assert api_key == "demo-api-key"

# Remove these names from the kernel's environment again
for key in config_keys:
    os.environ.pop(key, None)